In [1]:
# проверим, что файл рабочий и выведем 3 строчки
with open("bible.txt", 'r') as bible:
    for i in range(3):
        print(bible.readline().strip())

In the beginning God created the heaven and the earth. And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.
And God said, Let there be light: and there was light.
And God saw the light, that it was good: and God divided the light from the darkness.


### 1. Реализация шага "Map"

In [2]:
%%writefile mapper.py
# сверху команда для сохранения данной ячейки в систему Jupyter

import sys


def mapper():
    # чтение блока данных происходит через sys.stdin
    for line in sys.stdin:
        for word in line.strip().split():
            if len(word) > 4:
                print(f"{word}\t1")


if __name__ == "__main__":
    mapper()

Writing mapper.py


### 2. Реализация шага "Reduce"

In [3]:
%%writefile reducer.py

import sys


def reducer():
    # между Map и Reduce срабатывает обязательная сортировка (Shuffle) и все данные с одинаковыми ключами будут идти подряд.
    # благодаря этому можно не сохранять огромный словарь всех ключей, а просто идти до тех пор, пока ключ не изменится.
    current_word = None
    current_count = 0

    for line in sys.stdin:
        word, count = line.strip().split('\t', 1)
        count = int(count)

        if word == current_word or current_word is None:
            # ключ не поменялся, поэтому прибавляем count к current_count
            current_count += count
        else:
            # ключ поменялся, поэтому выводим предыдущий ключ и сумму его значений
            print(f"{current_word}\t{current_count}")
            current_count = 1
            
        current_word = word

    # # в конце всё равно надо вывести последний ключ и его значение
    print(f"{current_word}\t{current_count}")


if __name__ == "__main__":
    reducer()

Writing reducer.py


### 3. Локальная проверка через "cat"

In [4]:
# cat входной файл | файл с маппером | сортировка по ключам | файл с редьюсером
! cat bible.txt | python3 mapper.py | sort | python3 reducer.py > local_result.txt
! cat local_result.txt

Aaron	205
Aaron,	95
Aaron.	11
Aaron:	2
Aaron;	5
Aaron?	1
Aaronites,	2
Aaron's	31
Abaddon,	1
Abagtha,	1
Abarim,	4
abase	3
abase.	1
abased,	1
abased;	3
abasing	1
abated	4
abated.	2
Abba,	3
Abdeel,	1
Abdi,	3
Abdiel,	1
Abdon	5
Abdon,	3
Abednego,	10
Abednego.	3
Abednego:	1
Abednego;	1
Abel,	4
Abel.	2
Abel:	1
Abelbethmaachah,	1
Abelmaim,	1
Abelmeholah	1
Abelmeholah,	2
Abelmizraim,	1
Abelshittim	1
Abez,	1
abhor	18
Abhor	1
abhorred	14
abhorred,	2
abhorrest	2
abhorreth	3
abhorreth,	1
abhorreth.	1
abhorring	1
Abia:	1
Abia;	1
Abiah	1
Abiah,	1
Abiah.	1
Abiah:	1
Abialbon	1
Abiasaph:	1
Abiathar	22
Abiathar,	5
Abiathar.	2
Abiathar:	1
Abiathar's	1
Abib,	1
Abib.	1
Abib:	1
Abib;	1
Abida,	1
Abidah,	1
Abidan	5
abide	72
abide,	3
abide.	1
abide:	1
Abide	5
abideth	29
abideth.	1
abiding	8
abiding.	1
Abiel	1
Abiel,	1
Abiel.	1
Abiezer	4
Abiezer,	2
Abiezer?	1
Abiezrite:	1
Abiezrites.	2
Abigail	12
Abigail,	3
Abigail.	1
Abigail:	1
Abihail	3
Abihail,	2
Abihail:	1
Abihu	3
Abihu,	9
Abihud,	1
Abijah	12
Abijah,	7
Abija

### 4. Запуск MapReduce-таска

In [5]:
# пришлось исправлять dockerfile.jupyter, т.к. стояла неправильная java_home 
!ls /usr/lib/jvm/

java-1.17.0-openjdk-amd64  java-1.8.0-openjdk-amd64
java-17-openjdk-amd64	   java-8-openjdk-amd64


In [6]:
# провеярем, что hdfs жив
! hdfs dfs -ls /

Found 2 items
drwxr-xr-x   - hadoop users          0 2026-09-26 16:46 /hadoop
drwxr-xr-x   - hadoop users          0 2026-09-26 15:38 /tmp


In [7]:
# создаём рабочую директорию и кладём входной файл
! hdfs dfs -mkdir -p /hadoop/hw1/task1/input
! hdfs dfs -put bible.txt /hadoop/hw1/task1/input
! hdfs dfs -ls /hadoop/hw1/task1/input

put: `/hadoop/hw1/task1/input/bible.txt': File exists
Found 1 items
-rw-r--r--   1 hadoop users    4047392 2026-09-26 16:46 /hadoop/hw1/task1/input/bible.txt


Надо предварительно удалить папку с результатом, потому что Hadoop не запустит процесс, если указанная папка не пустая.
-D mapreduce.job.reduces указывает сколько редьюсеров будет создано для выполнения таски.

In [8]:
%%bash
hdfs dfs -rm -r /hadoop/hw1/task1/output

hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-D mapreduce.job.reduces=3 \
-files mapper.py,reducer.py \
-mapper "python3 mapper.py" \
-reducer "python3 reducer.py" \
-input /hadoop/hw1/task1/input/ \
-output /hadoop/hw1/task1/output/

Deleted /hadoop/hw1/task1/output
packageJobJar: [/tmp/hadoop-unjar8473252498483874685/] [] /tmp/streamjob8034740565874562319.jar tmpDir=null


26/09/26 19:33:43 INFO DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.18.0.6:8032
26/09/26 19:33:43 INFO DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.18.0.6:8032
26/09/26 19:33:43 INFO JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1790436132414_0007
26/09/26 19:33:44 INFO FileInputFormat: Total input files to process : 1
26/09/26 19:33:44 INFO JobSubmitter: number of splits:2
26/09/26 19:33:44 INFO JobSubmitter: Submitting tokens for job: job_1790436132414_0007
26/09/26 19:33:44 INFO JobSubmitter: Executing with tokens: []
26/09/26 19:33:44 INFO Configuration: resource-types.xml not found
26/09/26 19:33:44 INFO ResourceUtils: Unable to find 'resource-types.xml'.
26/09/26 19:33:44 INFO YarnClientImpl: Submitted application application_1790436132414_0007
26/09/26 19:33:44 INFO Job: The url to track the job: http://resourcemanager:8088/proxy/applicat

In [9]:
! hdfs dfs -getmerge /hadoop/hw1/task1/output/ ./hadoop_result.txt
! cat hadoop_result.txt

(Which	2
(Yea,	1
(being	1
(from	1
(that	4
(they	3
(thou	1
(though	2
(whatsoever	1
Aaron	205
Aaron's	31
Aaron,	95
Aaron;	5
Abdiel,	1
Abel,	4
Abelbethmaachah,	1
Abelmizraim,	1
Abialbon	1
Abiathar's	1
Abib,	1
Abib;	1
Abiel.	1
Abiezer,	2
Abigail	12
Abihail,	2
Abihud,	1
Abijah:	1
Abijam	4
Abijam,	1
Abilene,	1
Abimelech's	2
Abimelech.	7
Abimelech:	1
Abinadab,	8
Abinoam.	1
Abishag	5
Abishalom.	2
Abital;	1
Abiud	1
Abiud;	1
Abner	44
Abner's	1
Abner,	10
Abner;	2
Abraham	120
Abraham?	2
Abram.	2
Abram:	1
Absalom,	21
Absalom;	1
Accept,	1
According	30
Accounting	1
Achaicus:	1
Achar,	1
Achim	1
Achim;	1
Achish	15
Achish,	4
Achsa.	1
Achzib:	1
Adah,	1
Adaiah	2
Adaiah,	6
Adami,	1
Adar.	2
Addar,	1
Adiel,	2
Adino	1
Adithaim,	1
Admah?	1
Adna,	1
Adna;	1
Adnah	1
Adnah,	1
Adonijah?	1
Adrammelech	1
Adummim,	2
Aeneas,	2
Aenon	1
Afterwards	2
Agabus,	1
Against	9
Agar.	1
Agrippa	6
Ahab.	3
Ahab:	3
Aharhel	1
Ahasai,	1
Ahasuerus'	1
Ahaz.	3
Ahaziah,	4
Ahaziah,)	1
Ahiezer,	1
Ahihud	1
Ahijah.	2
Ahikam.	1
Ahikam:	1
Ahimaa

Видим, что "Aaron" насчитался 205 раз, а "Aaron," 95 раз в обоих выводах. Значит всё сделано верно.